# Energy, Bandwidth, and Aspect Ratio: FZP Cascade vs Optimized Cascade

This notebook is the Fig. 2(a) comparison for a **10-element intermediate-field Fresnel zone-plate cascade** versus a **10-element optimized cascade**:

1. **Bandwidth vs. Energy** — how efficiency degrades as the illumination bandwidth increases, across X-ray energies
2. **Aspect Ratio (Thickness) vs. Energy** — how efficiency depends on element thickness (and hence aspect ratio) at each energy

The FZP cascade uses coinciding foci: the upstream plate fills the cascade aperture and downstream radii follow the first-order cone. Load results from:

- `paper/sweeps/configs/fzp_cascade_bandwidth_energy_sweep.py`
- `paper/sweeps/configs/fzp_cascade_thickness_energy_sweep.py`

In the saved arrays, `fzp_*` is the zone-plate **cascade** (not a single zone plate).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(repo_root))

from src.util import width_central_peak

In [ ]:
matplotlib.rcParams['figure.dpi'] = 200
matplotlib.rcParams.update({'font.size': 24})

## Load sweep data

Prefer stable untimestamped files in `paper_data/`. If those are missing, fall back to the latest timestamped run. The efficiency arrays are extracted for both the optimized cascades and the FZP cascade.

In [ ]:
def resolve_sweep_files(data_dir, prefix, study_key):
    params_path = data_dir / f"{prefix}_params.npy"
    results_path = data_dir / f"{prefix}_results.npz"
    arrays_path = data_dir / f"{prefix}_sweep_arrays.npy"

    if not params_path.exists() or not results_path.exists():
        param_candidates = sorted(data_dir.glob(f"{prefix}_params_*.npy"))
        if not param_candidates:
            raise FileNotFoundError(
                f"No {prefix} results in {data_dir}. Run the matching sweep first:\n"
                f"  python paper/sweeps/run_sweep.py --study {study_key} --save-dir paper_data"
            )
        params_path = param_candidates[-1]
        stamp = params_path.name[len(prefix) + len("_params_"):-len(".npy")]
        results_path = data_dir / f"{prefix}_results_{stamp}.npz"
        arrays_path = data_dir / f"{prefix}_sweep_arrays_{stamp}.npy"

    return params_path, results_path, arrays_path


def load_sweep(data_dir, prefix, study_key):
    params_path, results_path, arrays_path = resolve_sweep_files(data_dir, prefix, study_key)
    params = np.load(params_path, allow_pickle=True).item()
    results = np.load(results_path, allow_pickle=True)
    if arrays_path.exists():
        sweep_arrs = np.load(arrays_path, allow_pickle=True).item()
    else:
        sweep_arrs = dict(params.get("axes", {}))
        for key in ("bandwidths", "thicknesses", "energies"):
            if key in params and key not in sweep_arrs:
                sweep_arrs[key] = np.asarray(params[key])
    print(params_path.name)
    print(results_path.name)
    return params, results, sweep_arrs

In [ ]:
path = repo_root / "paper_data"

bw_params, bw_results, bw_sweep_arrs = load_sweep(
    path, "fzp_cascade_bandwidth_energy_sweep", "fzp_cascade_bandwidth_energy"
)
ar_params, ar_results, ar_sweep_arrs = load_sweep(
    path, "fzp_cascade_thickness_energy_sweep", "fzp_cascade_thickness_energy"
)

In [ ]:
bw_opt_intensities = bw_results["opt_intensities"]
bw_opt_efficiencies = bw_results["opt_efficiencies"].T

bw_fzp_intensities = bw_results["fzp_intensities"]
bw_fzp_efficiencies = bw_results["fzp_efficiencies"].T

bandwidths = bw_sweep_arrs["bandwidths"]
bw_energies = bw_sweep_arrs["energies"]

ar_opt_intensities = ar_results["opt_intensities"]
ar_opt_efficiencies = ar_results["opt_efficiencies"].T

ar_fzp_intensities = ar_results["fzp_intensities"]
ar_fzp_efficiencies = ar_results["fzp_efficiencies"].T

thicknesses = ar_sweep_arrs["thicknesses"]
ar_energies = ar_sweep_arrs["energies"]

energies = ar_energies

In [ ]:
# recompute the efficiencies to exclude very wide focal spots
for i in range(len(bandwidths)):
    for j in range(len(energies)):

        Nx = bw_params['Nx']

        opt_intensity = bw_opt_intensities[i, j]
        fzp_intensity = bw_fzp_intensities[i, j]

        opt_width = width_central_peak(opt_intensity, 1e-2)
        fzp_width = width_central_peak(fzp_intensity, 1e-2)

        opt_efficiency = np.sum(opt_intensity[Nx//2 - opt_width//2 : Nx//2 + opt_width//2]) / Nx
        fzp_efficiency = np.sum(fzp_intensity[Nx//2 - fzp_width//2 : Nx//2 + fzp_width//2]) / Nx

        if opt_width > Nx / 10:
            opt_efficiency = 1 / Nx
        if fzp_width > Nx / 10:
            fzp_efficiency = 1 / Nx

        bw_opt_efficiencies[j, i] = opt_efficiency
        bw_fzp_efficiencies[j, i] = fzp_efficiency

In [ ]:
# recompute the efficiencies to exclude very wide focal spots
for i in range(len(thicknesses)):
    for j in range(len(energies)):

        Nx = bw_params['Nx']

        opt_intensity = ar_opt_intensities[i, j]
        fzp_intensity = ar_fzp_intensities[i, j]

        opt_width = width_central_peak(opt_intensity, 1e-2)
        fzp_width = width_central_peak(fzp_intensity, 1e-2)

        opt_efficiency = np.sum(opt_intensity[Nx//2 - opt_width//2 : Nx//2 + opt_width//2]) / Nx
        fzp_efficiency = np.sum(fzp_intensity[Nx//2 - fzp_width//2 : Nx//2 + fzp_width//2]) / Nx

        if opt_width > Nx / 10:
            opt_efficiency = 1 / Nx
        if fzp_width > Nx / 10:
            fzp_efficiency = 1 / Nx

        ar_opt_efficiencies[j, i] = opt_efficiency
        ar_fzp_efficiencies[j, i] = fzp_efficiency

In [ ]:
max_eff = max([bw_opt_efficiencies.max(), ar_opt_efficiencies.max()])

In [ ]:
BW, E = np.meshgrid(bandwidths, energies/1e3)
T, E = np.meshgrid(thicknesses / ar_params['min_feature_size'], energies/1e3)

fig = plt.figure(figsize=(9, 9.5))

gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.15],
                      wspace=0.25, hspace=0.16,
                      left=0.1, right=0.92, bottom=0.15, top=0.91)

ax_tl = fig.add_subplot(gs[0, 0])
ax_tr = fig.add_subplot(gs[0, 1])
ax_bl = fig.add_subplot(gs[1, 0])
ax_br = fig.add_subplot(gs[1, 1])
cax   = fig.add_subplot(gs[:, 2])

ax = np.array([[ax_tl, ax_tr], [ax_bl, ax_br]])

ax[0,0].pcolormesh(BW, E, bw_fzp_efficiencies, shading='auto', cmap='gray', vmin=0.0, vmax=max_eff)
im = ax[1,0].pcolormesh(BW, E, bw_opt_efficiencies, shading='auto', cmap='gray', vmin=0.0, vmax=max_eff)

ax[0,1].pcolormesh(T, E, ar_fzp_efficiencies, shading='auto', cmap='gray', vmin=0.0, vmax=max_eff)
im = ax[1,1].pcolormesh(T, E, ar_opt_efficiencies, shading='auto', cmap='gray', vmin=0.0, vmax=max_eff)

fig.text(0.46, 0.92, 'zone plate cascade', fontsize=28, ha='center', va='bottom')
fig.text(0.46, 0.505, 'optimized cascade', fontsize=28, ha='center', va='bottom')

cbar = fig.colorbar(im, cax=cax)
cbar.ax.tick_params(labelsize=24)
cbar.set_label("efficiency", size=26)

for a in ax.flat:
    a.semilogx()
    a.tick_params(labelsize=24)

ax[1,0].set_xlabel("bandwidth", fontsize=26)
ax[0,0].set_ylabel("energy [keV]", fontsize=26)
ax[1,0].set_ylabel("energy [keV]", fontsize=26)
ax[1,1].set_xlabel("aspect ratio", fontsize=26)

ax[0,0].xaxis.set_visible(False)
ax[0,1].xaxis.set_visible(False)
ax[0,1].yaxis.set_visible(False)
ax[1,1].yaxis.set_visible(False)

plt.show()